# Dependencies

In [185]:
import sys
import pandas as pd
import argparse
import os
from unittest.mock import patch, MagicMock, call

In [186]:
sys.path.insert(0, '/Users/karenzhu/Program/SAT2')
from sat.scripts.utils.misc import talk_to_me, make_output_dir
from sat.scripts.utils.structure import pdb_to_structure_object, write_structure_subset


# Temp data

In [187]:
class Args:
    pass

# Functions

In [188]:
def parse_chainsaw_file(chainsaw_file):
    """
    Given text file of chainsaw results, read the results into a table and 
    return a dictionary of structure name:domain boundary key value pairs.
    If a structure has zero domains, it will not be included in the dictionary.

    - chainsaw_file: chainsaw results text file, containing chain_id, sequence_md5, nres, ndom, chopping, confidence, time_sec
    - domains_dict: dictionary of structure name: domain boundaries
    """
    chainsaw_table = pd.read_csv(chainsaw_file, sep='\t')
    
    talk_to_me("Number of structures with no domains: " + str(chainsaw_table['chopping'].isnull().sum()))
    talk_to_me("Number of structures with at least 1 domain: " + str(chainsaw_table['chopping'].notnull().sum()))

    chainsaw_table_filtered = chainsaw_table.dropna(subset=['chopping'])
    domains_dict = chainsaw_table_filtered.set_index('chain_id')['chopping'].to_dict()

    return domains_dict

In [189]:
def format_chainsaw_domains(structure_name, domain_dict):    
    """
    Given the structure name and domain dictionary, retrieve the domain boundaries
    for the given structure. Use the domain boundaries to create a domain_residues_list. 
    Each list in domain_residues_list is some iterable/list of numbers, where each number is the position
    of one of the residues to keep.

    - structure_name: name of the structure
    - domain_dict: dictionary of structure name: domain boundaries
    - domain_residues_list: a list of lists of domain residues corresponding to the structure_name. 
    """
    
    def parse_domain(domain):
        domain_start = int(domain.split('-')[0])
        domain_end = int(domain.split('-')[1])
        domain_residues = [i for i in range(domain_start, domain_end+1)]
        return domain_residues 

    domain_boundaries = domain_dict.get(structure_name) 
    domain_list = domain_boundaries.split(',')
    domain_residues_list = []
    for domain in domain_list:
        if '_' not in domain:
            domain_residues_list.append(parse_domain(domain))
        elif '_' in domain:
            subdomain_list = domain.split('_')
            subdomain_residues_list = []
            for subdomain in subdomain_list:
                subdomain_residues_list = subdomain_residues_list + parse_domain(subdomain)
            domain_residues_list.append(subdomain_residues_list)

                
    return domain_residues_list

In [208]:
def get_pdb_filename(file_path):
    """
    Give the path to a pdb file, extract and output the name of the pdb file.

    - file_path: path to pdb file
    """
    if file_path.endswith(".pdb"):
        input_file_name = os.path.splitext(os.path.basename(file_path))[0]
        return input_file_name
    else:
        talk_to_me("This is not a pdb file")
        return
 


In [209]:
def struc_extract_residues(pdb_file_path, domain_residues_list, min_domain_length, output_dir):
    """
    For the pdb file, extract and output the domains (pdb format) that meet the min_domain_length requirement.
    
    - pdb_file_path: path to pdb file
    - domain_residues_list: a list of lists of domain residues
    - min_domain_length: The length cutoff for the domains. 
                         If domain length < min_domain_length, the domain will not be written out to file.
    - output_dir: directory to output the pdb files
    """
    structure = pdb_to_structure_object(pdb_file_path, structure_name="structure")
    for domain_residues in domain_residues_list:
        domain_start = min(domain_residues)
        domain_end = max(domain_residues)

        if len(domain_residues) < min_domain_length:
            continue
        else:
            input_file_name = get_pdb_filename(pdb_file_path)
            output_file_name = input_file_name + '_domain' + '_'+ str(domain_start) + '_' +  str(domain_end) + '.pdb'

            file_path = os.path.join(output_dir, output_file_name) 
            write_structure_subset(structure, residues_to_keep=domain_residues, outfile=file_path)
    return

# Main

In [210]:
#required input should be chainsaw_file, structure_file, min_domain_length, and outfile_dir
def struct_get_domains_main(args):
    domain_dict = parse_chainsaw_file(args.chainsaw_file_path)
    structure_name = get_pdb_filename(args.structure_file_path)
    domain_residues_list = format_chainsaw_domains(structure_name=structure_name, domain_dict=domain_dict)
    make_output_dir(args.outfile_dir, is_dir=True)
    struc_extract_residues(args.structure_file_path, domain_residues_list, args.min_domain_length, args.outfile_dir)
    return

In [207]:
args = Args()
BASE="/Users/karenzhu/Desktop/Learning_SAT"

args.chainsaw_file_path = f"{BASE}/chainsaw_data/first_60_arch_combined_chainsaw.txt"
args.structure_file_path = f"{BASE}/first_60_arch_pdbs/AB537968__BAJ06120.1__X__00010.pdb"
args.outfile_dir = f"{BASE}/output2"
args.min_domain_length = 20

struct_get_domains_main(args)


ipykernel_launcher.py: Number of structures with no domains: 4
ipykernel_launcher.py: Number of structures with at least 1 domain: 55


In [201]:
# #read in the file
#chainsaw_table = pd.read_csv(args.chainsaw_file, sep='\t')
#display(chainsaw_table)

# Tests

In [195]:
def test_single_domain_formatting():
    domain_dict = {'structure1': "3-8"}
    expected = [[3, 4, 5, 6, 7, 8]]
    assert format_chainsaw_domains('structure1', domain_dict) == expected


In [196]:
def test_multiple_domains_formatting():
    domain_dict = {'structure1': "3-8, 11-15"}
    expected = [[3, 4, 5, 6, 7, 8], [11, 12, 13, 14, 15]]
    assert format_chainsaw_domains('structure1', domain_dict) == expected

In [197]:
def test_single_domain_with_subdomain_formatting():
    domain_dict = {'structure1': "3-8_11-15"}
    expected = [[3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15]]
    assert format_chainsaw_domains('structure1', domain_dict) == expected

In [198]:
def test_complex_domains_with_subdomains_formatting():
    domain_dict = {'structure1': "3-8_11-15, 20-25, 30-32_40-45"}
    expected = [[3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15], [20, 21, 22, 23,24,25], [30,31,32,40,41,42,43,44,45]]
    assert format_chainsaw_domains('structure1', domain_dict) == expected

In [199]:
@patch('__main__.pdb_to_structure_object')
@patch('__main__.get_filename')
@patch('__main__.write_structure_subset')
#@patch('__main__.os.path.join')
def test_domain_length_less_than_min(mock_write_structure_subset, mock_get_filename, mock_pdb_to_structure_object):

    mock_structure = MagicMock()
    mock_pdb_to_structure_object.return_value = mock_structure
    mock_get_filename.return_value = 'test_structure'
    #mock_os_path_join.side_effect = lambda *args: '/'.join(args)
    
    pdb_file_path = 'test.pdb'
    domain_residues_list = [
        [1],
        [1,2],
        [1,2,3],
        [1,2,3,4],
        [2,3,7,8,9]
    ]
    min_domain_length = 3
    output_dir = 'output'

    struc_extract_residues(pdb_file_path, domain_residues_list, min_domain_length, output_dir)

    expected_calls = [
        call(mock_structure, residues_to_keep=[1,2,3], outfile='output/test_structure_domain_1_3.pdb'),
        call(mock_structure, residues_to_keep=[1,2,3,4], outfile='output/test_structure_domain_1_4.pdb'),
        call(mock_structure, residues_to_keep=[2,3,7,8,9], outfile='output/test_structure_domain_2_9.pdb')
    ]
    mock_write_structure_subset.assert_has_calls(expected_calls, any_order=False)
    assert mock_write_structure_subset.call_count == 3

    return

    

In [200]:
test_single_domain_formatting()
test_multiple_domains_formatting()
test_single_domain_with_subdomain_formatting()
test_complex_domains_with_subdomains_formatting()
test_domain_length_less_than_min()
